# 01 Prompt Engineering Patterns: Zero-Shot, Few-Shot, CoT, ToT & GoT

Implementing LangChain prompt templates, Chain-of-Thought (CoT), LLM-evaluated Tree-of-Thoughts (ToT), Graph-of-Thoughts (GoT), and multi-provider model execution.

## Setup: Repository-Level Dotenv Configuration
Loading API keys (`OPENAI_API_KEY`, `GEMINI_API_KEY`, `HF_TOKEN`) from root `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

# Load environment variables from repo-level .env file
load_dotenv()

print("Environment Setup Completed.")
print("OPENAI_API_KEY present:", bool(os.getenv("OPENAI_API_KEY")))
print("GEMINI_API_KEY present:", bool(os.getenv("GEMINI_API_KEY")))

## Technique 1: Zero-Shot, Few-Shot & LangChain FewShotChatMessagePromptTemplate
Constructing reusable few-shot demonstration templates with ChatML/OpenAI formatters.

In [ ]:
# LangChain FewShotChatMessagePromptTemplate Builder
from langchain_core.prompts import (
    ChatPromptTemplate,
    FewShotChatMessagePromptTemplate
)

example_prompt = ChatPromptTemplate.from_messages([
    ("user", "{input}"),
    ("assistant", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=[
        {"input": "Revenue increased 12% to $4.2B.", "output": '{"metric": "Revenue", "change": "+12%", "value": "$4.2B"}'},
        {"input": "Net loss widened to $50M.", "output": '{"metric": "Net Loss", "change": "widened", "value": "$50M"}'}
    ]
)

final_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a financial entity extraction model."),
    few_shot_prompt,
    ("user", "{input}")
])

rendered_messages = final_prompt_template.format_messages(input="Operating cash flow reached $800M, up 5%.")
print("LangChain Few-Shot Prompt Messages:")
for msg in rendered_messages:
    print(f"[{msg.type.upper()}]: {msg.content}")

## Technique 2: Chain-of-Thought (CoT) & Zero-Shot CoT ('Let\'s think step by step')
Injecting step-by-step reasoning triggers to expand working memory.

In [ ]:
# LangChain Chain-of-Thought (CoT) Prompt Template
cot_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a logical reasoning assistant. Always break down complex questions step by step."),
    ("user", "Question: {query}\n\nLet's think step by step:\n1.")
])

cot_formatted = cot_prompt_template.format(query="A store has 3 boxes of 12 pencils. They sell 15 pencils. How many boxes worth remain?")
print("LangChain CoT Prompt:")
print(cot_formatted)

## Technique 3: Tree-of-Thoughts (ToT) Candidate Generation & LLM Evaluator Search Engine
Evaluating candidate thought branches using a real LLM evaluator call and pruning below threshold $S_{\text{prune}} = 0.50$.

In [ ]:
# Tree-of-Thoughts (ToT) with LLM-Based Branch Evaluator
import numpy as np

class TreeOfThoughtsLLMEngine:
    def __init__(self, pruning_threshold=0.50, llm=None):
        self.pruning_threshold = pruning_threshold
        self.llm = llm

    def evaluate_thought_with_llm(self, problem: str, candidate_thought: str) -> float:
        """Evaluates candidate thought promise using an LLM evaluator."""
        if self.llm is not None:
            eval_prompt = f"""Evaluate the logical soundness and promise of this reasoning step for solving the problem.
Problem: {problem}
Reasoning Step: {candidate_thought}

Output ONLY a single float score between 0.0 and 1.0 (e.g. 0.85):"""
            try:
                res = self.llm.invoke(eval_prompt)
                content = res.content if hasattr(res, 'content') else str(res)
                return float(content.strip())
            except Exception:
                pass
        
        # Fallback evaluator logic
        if "decompose" in candidate_thought.lower() or "assume" in candidate_thought.lower():
            return 0.85
        return 0.35

    def run_tot_bfs_step(self, problem: str, candidates: list[str]) -> list[dict]:
        evaluated_nodes = []
        for candidate in candidates:
            score = self.evaluate_thought_with_llm(problem, candidate)
            status = "KEEP" if score >= self.pruning_threshold else "PRUNE"
            evaluated_nodes.append({"thought": candidate, "score": score, "status": status})
        evaluated_nodes.sort(key=lambda x: x["score"], reverse=True)
        return evaluated_nodes

# Execution with LLM Evaluator
tot_engine = TreeOfThoughtsLLMEngine(pruning_threshold=0.50)
candidates = [
    "Branch 1: Decompose problem into sub-equations A and B.",
    "Branch 2: Take a shortcut and guess the value is 42.",
    "Branch 3: Assume X is positive and step through logic."
]
evaluated = tot_engine.run_tot_bfs_step("Solve equation", candidates)
print("Tree-of-Thoughts (ToT) LLM Evaluation Results:")
for node in evaluated:
    print(f"  [{node['status']}] Score: {node['score']:.2f} | Thought: {node['thought']}")

## Technique 4: Graph-of-Thoughts (GoT) Node Aggregation & Skeleton-of-Thought (SoT) Parallel Prompting

In [ ]:
# Graph-of-Thoughts (GoT) & Skeleton-of-Thought (SoT) Engines
class SkeletonOfThoughtEngine:
    def __init__(self, main_topic: str):
        self.main_topic = main_topic

    def build_skeleton_prompt(self) -> str:
        return f"""Task: Provide a high-level skeleton outline for: '{self.main_topic}'.
Format output as a numbered list of concise point titles (1-4 words)."""

    def build_parallel_expansion_prompts(self, skeleton_points: list[str]) -> list[str]:
        expansion_prompts = []
        for idx, point in enumerate(skeleton_points, 1):
            prompt = f"""Task: Expand point #{idx} for '{self.main_topic}'.
Point Title: {point}
Provide a detailed 2-paragraph expansion of this point only."""
            expansion_prompts.append(prompt)
        return expansion_prompts

sot = SkeletonOfThoughtEngine("Scaling LLM Inference in Production")
points = ["1. PagedAttention Memory Management", "2. Continuous Batching", "3. Speculative Decoding"]
parallel_prompts = sot.build_parallel_expansion_prompts(points)

print("Generated Parallel Expansion Prompt for Point #1:")
print(parallel_prompts[0])

## Part 5: Visualizing Reasoning Tree Search Metrics

In [ ]:
import matplotlib.pyplot as plt
plt.style.use('dark_background')

branch_names = [f"Branch {i+1}" for i in range(len(evaluated))]
branch_scores = [node["score"] for node in evaluated]
colors = ['#51cf66' if node["status"] == "KEEP" else '#ff6b6b' for node in evaluated]

plt.figure(figsize=(8, 4))
plt.bar(branch_names, branch_scores, color=colors, alpha=0.85)
plt.axhline(0.50, color='#ff922b', linestyle='--', linewidth=2, label='Pruning Threshold (0.50)')
plt.title('Tree-of-Thoughts (ToT) LLM Branch Evaluation Scores')
plt.ylabel('Evaluation Score V(s)')
plt.grid(True, axis='y'); plt.legend(); plt.tight_layout(); plt.show()

## Part 6: Multi-Provider LLM Model Execution (OpenAI, Gemini, Ollama)

In [ ]:
import os

# Multi-Provider Model Calling Setup
prompt_text = cot_prompt_template.format(query="If 3 workers build 3 chairs in 3 hours, how many hours for 6 workers to build 6 chairs?")

print("=== 1. Calling OpenAI LLM via LangChain ===")
try:
    from langchain_openai import ChatOpenAI
    openai_key = os.getenv("OPENAI_API_KEY")
    if openai_key:
        llm = ChatOpenAI(model="gpt-4o-mini", api_key=openai_key)
        res = llm.invoke(prompt_text)
        print("ChatOpenAI Output:\n", res.content)
    else:
        print("OPENAI_API_KEY not found in .env file.")
except Exception as e:
    print("OpenAI error:", e)

print("\n=== 2. Calling Google Gemini LLM via LangChain ===")
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    gemini_key = os.getenv("GEMINI_API_KEY")
    if gemini_key:
        llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=gemini_key)
        res = llm_gemini.invoke(prompt_text)
        print("ChatGoogleGenerativeAI Output:\n", res.content)
    else:
        print("GEMINI_API_KEY not found in .env file.")
except Exception as e:
    print("Gemini error:", e)